### Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir('C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import gc
import numpy as np
import pandas as pd
import config
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, MinMaxScaler
from src.utils import *
from sklearn.utils.class_weight import compute_class_weight
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from tqdm.notebook import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from scipy.special import softmax
from lightgbm import LGBMClassifier, log_evaluation,early_stopping
import optuna
from optuna.integration import LightGBMPruningCallback
from src.fe import *
pd.set_option('display.max_columns', None)

import logging
logging.getLogger("mlflow.utils.environment").setLevel(logging.ERROR)

In [3]:
import warnings
warnings.filterwarnings("ignore", category=pd.errors.ChainedAssignmentError)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [4]:
# suppress pytorch lightning completely
os.environ["PYTORCH_LIGHTNING_DISABLE_TIPS"] = "1"
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

for name in logging.root.manager.loggerDict:
    if "lightning" in name or "pytorch" in name:
        logging.getLogger(name).setLevel(logging.ERROR)

warnings.filterwarnings("ignore", module="pytorch_lightning.*")
warnings.filterwarnings("ignore", module="lightning.*")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

### Prepare data

In [5]:
train = read_csv_file('data/raw/train.csv')
test = read_csv_file('data/raw/test.csv')

test_ids = test.id
train_ids = train.id
train[config.TARGET] = train[config.TARGET].map(config.TARGET_MAP)

y = train[config.TARGET].astype(int)

Reading file from: data/raw/train.csv
Reading file from: data/raw/test.csv


In [8]:
os.listdir('artifacts/submissions')

['blend_catgbm_seed42_v9_lr_seed42_v21_hist_seed42_v4_thresh_tuned_submission.csv',
 'blend_catgbm_v9_lr_v21_hist_v4_resnet_v4_xgbm_v1_thresh_tuned_submission.csv',
 'blend_catV10_lrV21_histV4_resnetV4_xgbV1_rmlpV3_TT_submission.csv',
 'blend_catV10_lrV21_histV4_resnetV4_xgbV1_rmlpV5_lgbmV1_test_proba_TT_sub.csv',
 'blend_catV10_lrV21_histV4_resnetV4_xgbV1_rmlpV5_submission.csv',
 'blend_catV10_lrV21_histV4_resnetV4_xgbV1_rmlpV5_TT_submission.csv',
 'blend_catV10_lrV21_histV4_resnetV4_xgbV2_rmlpV5_TT_submission.csv',
 'blend_catV9_lrV21_histV4_resnetV4_xgbmV1_rmlpV5_TT_submission.csv',
 'blend_catV9_lrV21_histV4_xgbmV1_rmlpV5_TT_submission.csv',
 'blend_cat_v9_lr_v21_hist_v4_resnet_v4_xgbm_v1_rmlp_v3_thresh_tuned_submission.csv',
 'blend_lr_v21_xgbm_v1_thresh_tuned_submission.csv',
 'Logistic_seed42_v21_Catgbm_seed42_v5_blend_thresh_tuned_submission.csv',
 'Logistic_seed42_v21_Catgbm_seed42_v9_blend_thresh_tuned_submission.csv',
 'logistic_seed_42_v21_thresh_tuned.csv',
 'realMLP_seed4

In [146]:
M=train[config.BASE_NUM_COLS].max()
def FE(df): 
    
    for c in config.BASE_NUM_COLS:
        for k in range(-4,4):
            df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')

        if M[c]<10:
            df[c]=df[c].round(3)
        elif M[c]<100:
            df[c]=df[c].round(2)
        else:
            df[c]=df[c].round(1)
    return df 

for df in [train,test]:
    df=FE(df)

In [147]:
DROP=[c for c in test.columns if test[c].nunique()==1]
print(f"DROP:{DROP}")
train.drop(DROP,axis=1,inplace=True)
test.drop(DROP,axis=1,inplace=True)

CATEGORY=config.BASE_CAT_COLS+[c for c in test.columns if 'digit' in c]
for c in CATEGORY:

    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))
    

FEATURES=CATEGORY+config.BASE_NUM_COLS

DROP:['soil_ph_digit1', 'soil_ph_digit2', 'soil_ph_digit3', 'soil_moisture_digit2', 'soil_moisture_digit3', 'organic_carbon_digit1', 'organic_carbon_digit2', 'organic_carbon_digit3', 'electrical_conductivity_digit1', 'electrical_conductivity_digit2', 'electrical_conductivity_digit3', 'temperature_c_digit2', 'temperature_c_digit3', 'humidity_digit2', 'humidity_digit3', 'sunlight_hours_digit2', 'sunlight_hours_digit3', 'wind_speed_kmh_digit2', 'wind_speed_kmh_digit3', 'field_area_hectare_digit2', 'field_area_hectare_digit3', 'previous_irrigation_mm_digit3']


In [148]:
X = train.drop([config.TARGET, 'id'],axis=1)
y = train[config.TARGET]
n_classes=3

In [149]:
unique, counts = np.unique(train[config.TARGET].values, return_counts=True)
count_dict = dict(zip(unique, counts))
avg_count = len(train) / len(unique)
weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
sample_weights = np.array([weights_dict[y] for y in train[config.TARGET]])

In [150]:
def lgb_eval_metric(y_true, y_pred):
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 3)

    y_pred = np.argmax(y_pred, axis=1)
    score = balanced_accuracy_score(y_true, y_pred)
    return "bacc", score, True

In [151]:
def objective(trial):

    params = {
        "boosting_type": "gbdt",
        "random_state": config.SEED,
        "n_jobs": -1,
        "verbosity": -1,
        "n_estimators": 6000,
        "subsample_for_bin": 100000,
        'max_bin': 15000,
        "subsample_freq": 1,
        'max_depth': 4,
        "num_leaves": 32,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-2, 50.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-2, 50.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
    }

    skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=config.SEED)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf_tune.split(X, y)):

        X_train = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]
        train_weights = sample_weights[train_idx]

        # target encoding
        te = TargetEncoder(target_type="multiclass", smooth="auto", cv=5, random_state=42)
        X_train_enc = te.fit_transform(X_train[FEATURES], y_train)
        X_val_enc = te.transform(X_val[FEATURES])

        X_train_enc = pd.DataFrame(X_train_enc, index=X_train.index)
        X_val_enc = pd.DataFrame(X_val_enc, index=X_val.index)

        X_train = pd.concat([X_train, X_train_enc], axis=1)
        X_val = pd.concat([X_val, X_val_enc], axis=1)

        X_train = X_train.drop(config.BASE_CAT_COLS, axis=1)
        X_val = X_val.drop(config.BASE_CAT_COLS, axis=1)

        # train
        model = LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            sample_weight=train_weights,
            eval_metric=lgb_eval_metric,
            callbacks=[
                early_stopping(250, verbose=False),
                log_evaluation(100)
            ]
        )

        fold_score = balanced_accuracy_score(y_val, model.predict(X_val))
        fold_scores.append(fold_score)

        trial.report(fold_score, step=fold)

        if trial.should_prune():
            raise optuna.TrialPruned()

        if len(fold_scores) > 1:
            prev_median = np.median(fold_scores[:-1])
            if fold_score < prev_median:
                raise optuna.TrialPruned()

    return np.mean(fold_scores)

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=config.SEED),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=0,
        interval_steps=1
    )
)

study.optimize(objective, n_trials=100, show_progress_bar=True)

In [ ]:
pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

print(f"\n{'='*50}")
print(f"Total trials: {len(study.trials)}")
print(f"Completed: {len(completed_trials)}")
print(f"Pruned (saved): {len(pruned_trials)}")
print(f"{'='*50}")
print(f"✅ Best score: {study.best_value:.6f}")
print(f"✅ Best params:")
for k, v in study.best_params.items():
    print(f"   {k:25s} = {v}")

lgb_params = {
    "boosting_type": "gbdt",
    "random_state": config.SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "n_estimators": 6000,    # full trees for main training
    "subsample_for_bin": 100000,
    "subsample_freq": 1,
    **study.best_params            # inject all tuned params
}

print(f"\n✅ lgb_params ready for full CV training!")
print(lgb_params)